# Seattle Collision Severity Regression

Six regressors compare historical grid-cell severity using the same predictors, split, and cross-validation folds. Both raw and stabilized severity remain candidate targets. Each fit learns the stabilization prior from its own training rows.

Previous scores and model rankings have been removed because they used a prior shared across validation folds. Corrected full-search results are pending. Small self-checks verify the evaluation wiring; they do not establish model performance.

This is a historical cell-level comparison, not evidence of future forecasting performance or superiority across seeds.


## Context & Methods

The modeling unit is an occupied spatial grid cell. The target is collision severity aggregated within each cell. Candidate regressors follow the original notebook's progression: a decision tree, a random forest, XGBoost, LightGBM, CatBoost, and a PyTorch multilayer perceptron.

### Key Assumptions

- Rows from the partial 2026 calendar year are excluded so every included year is complete.
- This is a historical, cell-level severity model. It does not claim to forecast a future year.
- Target-derived columns are explicitly excluded from the feature matrix.
- Each cross-validation fit learns its severity prior from that fold's training rows. Validation targets use the fitted prior; their labels do not enter training targets. Final held-out targets use the full-training refit's prior.
- Cross-validation selects the target and model; the test partition is used only for final evaluation.

## Data

### 1. Import libraries

In [ ]:
from importlib.metadata import version
from pathlib import Path
import platform

import geopandas as gpd
import pandas as pd
import numpy as np
import torch
from torch import nn

from scipy.stats import randint, loguniform, uniform
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.exceptions import NotFittedError
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_is_fitted, validate_data
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    RandomizedSearchCV,
    RepeatedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

notebook_versions = {
    package: version(package)
    for package in [
        'numpy', 'pandas', 'geopandas', 'scipy', 'scikit-learn',
        'xgboost', 'lightgbm', 'catboost', 'torch', 'matplotlib',
    ]
}
notebook_versions['python'] = platform.python_version()
print('Notebook package versions:', notebook_versions)

### 2. Load the processed collision data

In [ ]:
data_path = Path('../data/processed/Collision_Processed.parquet')
model_end_year = 2025

gdf = gpd.read_parquet(data_path)
gdf = gdf[gdf['YEAR'] <= model_end_year].copy()

print(f'Using {len(gdf):,} collision records through {model_end_year}.')
gdf.head()

### 3. Aggregate collisions into spatial grid cells

In [ ]:
grid = gdf.groupby(['lat_bin', 'lon_bin']).agg(
    collision_count=('COLDETKEY', 'count'),
    severity_sum=('SEVERITY', 'sum'),
    mean_severity=('SEVERITY', 'mean'),
    mean_injuries=('INJURIES', 'mean'),
    mean_serious=('SERIOUSINJURIES', 'mean'),
    mean_fatalities=('FATALITIES', 'mean'),
    avg_vehicle_count=('VEHCOUNT', 'mean'),
    avg_pedestrian_count=('TOTAL_PED', 'mean'),
    cell_lat=('Y', 'mean'),
    cell_lon=('X', 'mean'),
    peak_hour=('HOUR', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    night_ratio=('HOUR', lambda x: ((x < 6) | (x >= 20)).mean()),
    weekend_ratio=('Day_of_Week', lambda x: x.isin(['Saturday', 'Sunday']).mean()),
    winter_ratio=('Season', lambda x: (x == 'Winter').mean()),
    summer_ratio=('Season', lambda x: (x == 'Summer').mean()),
    intersection_ratio=(
        'JUNCTIONTYPE',
        lambda x: x.str.contains('At Intersection', case=False, na=False).mean(),
    ),
).reset_index()

print(f'Created {len(grid):,} occupied grid cells.')
grid[['collision_count', 'severity_sum', 'mean_severity']].describe()

## Target Engineering

### 4. Create raw and stabilized severity targets

`mean_severity` can be unstable in cells with few collisions. Stabilization shrinks a cell toward the training population's severity rate with weight 10. The plot below uses the outer training partition's prior for description only.

Cross-validation receives severity sums and collision counts. `FoldTargetRegressor` learns a separate prior inside each fit, trains on the resulting targets, and calculates validation R² using that same prior. Searches use the adapter's score method rather than applying a generic scorer to the two-column target statistics. Final test evaluation uses the full-training refit's prior.


In [ ]:
class FoldTargetRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, regressor, target_name='severity_smoothed', smoothing=10):
        self.regressor = regressor
        self.target_name = target_name
        self.smoothing = smoothing

    def target_values(self, statistics):
        check_is_fitted(self, ['prior_'])
        count = statistics['collision_count']
        total = statistics['severity_sum']
        if self.target_name == 'mean_severity':
            return total / count
        if self.target_name == 'severity_smoothed':
            return (total + self.smoothing * self.prior_) / (count + self.smoothing)
        raise ValueError(f'Unknown severity target: {self.target_name}')

    def fit(self, X, y):
        self.prior_ = y['severity_sum'].sum() / y['collision_count'].sum()
        self.regressor_ = clone(self.regressor)
        self.regressor_.fit(X, self.target_values(y))
        return self

    def predict(self, X):
        check_is_fitted(self, ['regressor_'])
        return self.regressor_.predict(X)

    def score(self, X, y):
        return r2_score(self.target_values(y), self.predict(X))


train_idx, test_idx = train_test_split(
    grid.index,
    test_size=0.2,
    random_state=42,
)

assert set(train_idx).isdisjoint(test_idx)

m = 10
global_severity = (
    grid.loc[train_idx, 'severity_sum'].sum()
    / grid.loc[train_idx, 'collision_count'].sum()
)

grid['severity_smoothed'] = (
    grid['severity_sum'] + m * global_severity
) / (grid['collision_count'] + m)

print(f'Training-only global severity prior: {global_severity:.4f}')
grid[['mean_severity', 'severity_smoothed']].describe()

In [ ]:
check_X = pd.DataFrame({'x': np.arange(24), 'missing': [np.nan, 1, 2] * 8})
check_statistics = pd.DataFrame({
    'severity_sum': np.arange(24, dtype=float) + 1,
    'collision_count': np.arange(24) % 3 + 1,
})
check_train = np.arange(16)
check_validation = np.arange(16, 24)
check_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', DecisionTreeRegressor(max_depth=2, random_state=42)),
])
for check_target in ['mean_severity', 'severity_smoothed']:
    check_model = FoldTargetRegressor(check_pipeline, target_name=check_target)
    check_model.fit(check_X.iloc[check_train], check_statistics.iloc[check_train])
    check_predictions = check_model.predict(check_X.iloc[check_validation])
    check_training_targets = check_model.target_values(check_statistics.iloc[check_train])
    changed_statistics = check_statistics.copy()
    changed_statistics.loc[check_validation, 'severity_sum'] += 1000
    repeated_model = clone(check_model).fit(
        check_X.iloc[check_train], changed_statistics.iloc[check_train],
    )
    assert repeated_model.prior_ == check_model.prior_
    pd.testing.assert_series_equal(
        repeated_model.target_values(changed_statistics.iloc[check_train]), check_training_targets,
    )
    np.testing.assert_array_equal(repeated_model.predict(check_X.iloc[check_validation]), check_predictions)
    expected_targets = check_statistics.iloc[check_validation]['severity_sum'] / check_statistics.iloc[check_validation]['collision_count']
    if check_target == 'severity_smoothed':
        expected_targets = (
            check_statistics.iloc[check_validation]['severity_sum'] + 10 * check_model.prior_
        ) / (check_statistics.iloc[check_validation]['collision_count'] + 10)
    assert check_model.score(check_X.iloc[check_validation], check_statistics.iloc[check_validation]) == r2_score(expected_targets, check_predictions)
    small_search = GridSearchCV(
        clone(check_model), {'regressor__model__max_depth': [1, 2]},
        cv=KFold(2), scoring=None, error_score='raise',
    ).fit(check_X.iloc[check_train], check_statistics.iloc[check_train])
    assert np.isfinite(small_search.cv_results_['mean_test_score']).all()
    assert small_search.best_estimator_.prior_ == check_statistics.iloc[check_train]['severity_sum'].sum() / check_statistics.iloc[check_train]['collision_count'].sum()
    repeated_scores = cross_val_score(
        clone(check_model), check_X, check_statistics,
        cv=RepeatedKFold(n_splits=2, n_repeats=2, random_state=42),
        scoring=None, error_score='raise',
    )
    assert np.isfinite(repeated_scores).all()
print('Fold-local target self-check passed.')


In [ ]:
grid['mean_severity'].hist(bins=40)
grid['severity_smoothed'].hist(bins=40)

## Leakage-Safe Features

### 5. Document and validate the predictors

The features below do not contain `SEVERITY`, injuries, serious injuries, fatalities, collision count, severity aggregates, the old risk score, or the old risk label. Each estimator fits its imputer within its training fold. The neural network also fits a scaler and reserves an inner validation split for early stopping.

`avg_vehicle_count` and `avg_pedestrian_count` are historical involvement summaries, not components of the severity formula. They are appropriate for this historical model, but a future forecasting model must compute them only from a prior lookback period.

In [ ]:
features = [
    'cell_lat',
    'cell_lon',
    'peak_hour',
    'night_ratio',
    'weekend_ratio',
    'winter_ratio',
    'summer_ratio',
    'avg_vehicle_count',
    'avg_pedestrian_count',
    'intersection_ratio',
]

target_derived_features = {
    'SEVERITY', 'INJURIES', 'SERIOUSINJURIES', 'FATALITIES',
    'collision_count', 'severity_sum', 'mean_severity', 'mean_injuries',
    'mean_serious', 'mean_fatalities', 'severity_smoothed',
    'risk_score', 'risk_label', 'risk_level',
}
assert target_derived_features.isdisjoint(features)

In [ ]:
X_train = grid.loc[train_idx, features]
X_test = grid.loc[test_idx, features]

print(f'Training cells: {len(X_train):,}')
print(f'Test cells: {len(X_test):,}')
print(f'Constant selected features: {X_train.columns[X_train.nunique() <= 1].tolist()}')
target_statistics = grid[['severity_sum', 'collision_count']]


### 6. Select the severity target with training-only cross-validation

R² is used for target selection because the raw and stabilized targets have different scales. Repeated folds reduce the chance of selecting a target based on one unusually favorable split. The held-out test set is not consulted here.

In [ ]:
target_cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

target_baseline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(
        n_estimators=200,
        max_depth=14,
        min_samples_leaf=5,
        max_features=0.7,
        n_jobs=1,
        random_state=42,
    )),
])

target_results = []
for target_name in ['mean_severity', 'severity_smoothed']:
    scores = cross_val_score(
        FoldTargetRegressor(target_baseline, target_name=target_name),
        X_train,
        target_statistics.loc[train_idx],
        cv=target_cv,
        scoring=None,
        n_jobs=-1,
    )
    target_results.append({
        'target': target_name,
        'mean_cv_r2': scores.mean(),
        'std_cv_r2': scores.std(),
    })

target_comparison = (
    pd.DataFrame(target_results)
    .sort_values('mean_cv_r2', ascending=False)
    .reset_index(drop=True)
)
selected_target = target_comparison.loc[0, 'target']

print(f'Selected target: {selected_target}')
target_comparison

## Results

### 7. Define regression evaluation

Models are ranked by cross-validated training R². Test R², MAE, and RMSE are reported once for each fitted model as diagnostics. The selected production candidate is chosen from cross-validation rather than test performance.

In [ ]:
y_train = target_statistics.loc[train_idx]
y_test = target_statistics.loc[test_idx]

model_cv = KFold(n_splits=5, shuffle=True, random_state=42)
model_results = []
fitted_models = {}
test_predictions = {}


def evaluate_regressor(model_name, search):
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    y_test_values = best_model.target_values(y_test)

    result = {
        'model': model_name,
        'cv_r2': search.best_score_,
        'test_r2': r2_score(y_test_values, y_pred),
        'mae': mean_absolute_error(y_test_values, y_pred),
        'rmse': mean_squared_error(y_test_values, y_pred) ** 0.5,
    }
    model_results.append(result)
    fitted_models[model_name] = best_model
    test_predictions[model_name] = y_pred

    print('Best parameters:', search.best_params_)
    print('Cross-validation R²:', np.round(result['cv_r2'], 3))
    print('Test R²:', np.round(result['test_r2'], 3))
    print('Test MAE:', np.round(result['mae'], 3))
    print('Test RMSE:', np.round(result['rmse'], 3))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].scatter(y_test_values, y_pred, alpha=0.35, s=18)
    bounds = [min(y_test_values.min(), y_pred.min()), max(y_test_values.max(), y_pred.max())]
    axes[0].plot(bounds, bounds, linestyle='--', color='black')
    axes[0].set_title(f'{model_name}: Actual vs Predicted')
    axes[0].set_xlabel('Actual severity score')
    axes[0].set_ylabel('Predicted severity score')

    residuals = y_test_values - y_pred
    axes[1].hist(residuals, bins=30, edgecolor='black')
    axes[1].axvline(0, linestyle='--', color='black')
    axes[1].set_title(f'{model_name}: Residual Distribution')
    axes[1].set_xlabel('Actual - predicted severity')
    axes[1].set_ylabel('Grid cells')

    plt.tight_layout()
    plt.show()

    fitted_regressor = best_model.regressor_
    fitted_regressor = getattr(fitted_regressor, 'named_steps', {}).get('model', fitted_regressor)
    if hasattr(fitted_regressor, 'feature_importances_'):
        feature_importance = pd.Series(
            fitted_regressor.feature_importances_,
            index=features,
        ).sort_values(ascending=True)
        feature_importance.plot(kind='barh')
        plt.title(f'{model_name}: Feature Importance')
        plt.xlabel('Importance')
        plt.show()

    return best_model

### 8. Tune a Decision Tree Regressor

In [ ]:
decision_tree_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', DecisionTreeRegressor(random_state=42)),
])

param_grid = {
    'regressor__model__max_depth': [4, 6, 8, 10, None],
    'regressor__model__min_samples_split': [2, 10, 20],
    'regressor__model__min_samples_leaf': [1, 5, 10, 20],
    'regressor__model__criterion': ['squared_error', 'friedman_mse'],
    'regressor__model__ccp_alpha': [0.0, 0.0001, 0.001],
}

grid_search = GridSearchCV(
    FoldTargetRegressor(decision_tree_pipeline, target_name=selected_target),
    param_grid,
    cv=model_cv,
    scoring=None,
    n_jobs=-1,
    return_train_score=True,
)
grid_search.fit(X_train, y_train)

decision_tree_model = evaluate_regressor('Decision Tree', grid_search)

### 9. Tune a Random Forest Regressor

In [ ]:
random_forest_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestRegressor(random_state=42, n_jobs=1)),
])

param_distributions = {
    'regressor__model__n_estimators': [200, 300, 400, 600, 800],
    'regressor__model__max_features': ['sqrt', 'log2', 0.5, 0.8],
    'regressor__model__max_depth': [None, 8, 12, 16, 24],
    'regressor__model__min_samples_split': [2, 5, 10, 20],
    'regressor__model__min_samples_leaf': [1, 2, 4, 8, 12],
    'regressor__model__bootstrap': [True, False],
}

random_forest_search = RandomizedSearchCV(
    FoldTargetRegressor(random_forest_pipeline, target_name=selected_target),
    param_distributions=param_distributions,
    n_iter=25,
    cv=model_cv,
    scoring=None,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
)
random_forest_search.fit(X_train, y_train)

random_forest_model = evaluate_regressor('Random Forest', random_forest_search)

### 10. Tune an XGBoost Regressor

In [ ]:
xgboost_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        eval_metric='rmse',
        tree_method='hist',
        n_jobs=1,
        random_state=42,
    )),
])

param_distributions = {
    'regressor__model__n_estimators': randint(200, 801),
    'regressor__model__max_depth': randint(2, 7),
    'regressor__model__learning_rate': loguniform(0.01, 0.15),
    'regressor__model__min_child_weight': loguniform(1, 30),
    'regressor__model__gamma': loguniform(1e-4, 1),
    'regressor__model__subsample': uniform(0.65, 0.35),
    'regressor__model__colsample_bytree': uniform(0.65, 0.35),
    'regressor__model__reg_alpha': loguniform(1e-4, 10),
    'regressor__model__reg_lambda': loguniform(0.1, 100),
}

xgboost_search = RandomizedSearchCV(
    FoldTargetRegressor(xgboost_pipeline, target_name=selected_target),
    param_distributions=param_distributions,
    n_iter=25,
    cv=model_cv,
    scoring=None,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
)
xgboost_search.fit(X_train, y_train)

xgboost_model = evaluate_regressor('XGBoost', xgboost_search)

### 11. Tune a LightGBM Regressor

In [ ]:
lightgbm_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', LGBMRegressor(
        objective='regression',
        subsample_freq=1,
        verbosity=-1,
        n_jobs=1,
        random_state=42,
    )),
])

param_distributions = {
    'regressor__model__n_estimators': randint(200, 801),
    'regressor__model__learning_rate': loguniform(0.01, 0.15),
    'regressor__model__num_leaves': randint(15, 64),
    'regressor__model__max_depth': [-1, 3, 4, 5, 6, 8],
    'regressor__model__min_child_samples': randint(10, 81),
    'regressor__model__subsample': uniform(0.65, 0.35),
    'regressor__model__colsample_bytree': uniform(0.65, 0.35),
    'regressor__model__reg_alpha': loguniform(1e-4, 10),
    'regressor__model__reg_lambda': loguniform(0.1, 100),
}

lightgbm_search = RandomizedSearchCV(
    FoldTargetRegressor(lightgbm_pipeline, target_name=selected_target),
    param_distributions=param_distributions,
    n_iter=25,
    cv=model_cv,
    scoring=None,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
)
lightgbm_search.fit(X_train, y_train)

lightgbm_model = evaluate_regressor('LightGBM', lightgbm_search)

### 12. Tune a CatBoost Regressor

In [ ]:
catboost_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', CatBoostRegressor(
        loss_function='RMSE',
        verbose=0,
        allow_writing_files=False,
        thread_count=1,
        random_seed=42,
    )),
])

param_distributions = {
    'regressor__model__iterations': randint(300, 1001),
    'regressor__model__depth': randint(4, 9),
    'regressor__model__learning_rate': loguniform(0.01, 0.15),
    'regressor__model__l2_leaf_reg': loguniform(1, 30),
    'regressor__model__random_strength': loguniform(1e-3, 3),
    'regressor__model__bagging_temperature': uniform(0, 1),
}

catboost_search = RandomizedSearchCV(
    FoldTargetRegressor(catboost_pipeline, target_name=selected_target),
    param_distributions=param_distributions,
    n_iter=25,
    cv=model_cv,
    scoring=None,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
)
catboost_search.fit(X_train, y_train)

catboost_model = evaluate_regressor('CatBoost', catboost_search)

### 13. Tune a PyTorch regression neural network

The baseline compares hidden layers `(32,)`, `(64,)`, and `(64, 32)`, each with weight decay `0` or `0.001`. ReLU hidden layers feed one linear output in the original severity units without clipping. CPU training uses float32, seed 42, Adam with learning rate 0.001, batches of 128, and mean squared error.

Each fit reserves 15% of its rows for early stopping. Median imputation and standardization first use only the remaining 85%. Training stops after 25 epochs without a validation-loss improvement greater than 0.000001, up to 300 epochs. The network then restarts from the same seed, refits preprocessing on all rows supplied to `fit`, and trains for the selected epoch count. Outer validation folds and the test set do not determine stopping. Grid search runs serially across the same five outer folds as the tree models.

The [PyTorch training guide](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) describes the training and evaluation modes used below. The synthetic check verifies missing-value handling, prediction shape, repeatability, and rejection of prediction before fitting.

In [ ]:
class TorchRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, hidden_sizes=(32,), weight_decay=0.001, random_state=42):
        self.hidden_sizes = hidden_sizes
        self.weight_decay = weight_decay
        self.random_state = random_state

    def _new_network(self):
        torch.manual_seed(self.random_state)
        layers = []
        input_size = self.n_features_in_
        for hidden_size in self.hidden_sizes:
            layers.extend([nn.Linear(input_size, hidden_size), nn.ReLU()])
            input_size = hidden_size
        layers.append(nn.Linear(input_size, 1))
        model = nn.Sequential(*layers).to(device='cpu', dtype=torch.float32)
        optimizer = torch.optim.Adam(
            model.parameters(), lr=0.001, weight_decay=self.weight_decay,
        )
        generator = torch.Generator(device='cpu').manual_seed(self.random_state)
        return model, optimizer, generator

    @staticmethod
    def _preprocessor():
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('scaler', StandardScaler()),
        ])

    @staticmethod
    def _tensor(values):
        values = np.asarray(values, dtype=np.float32)
        assert np.isfinite(values).all(), 'Non-finite transformed inputs or targets'
        return torch.from_numpy(values)

    @staticmethod
    def _train_epoch(model, optimizer, generator, X, y):
        model.train()
        total_loss = 0.0
        order = torch.randperm(len(X), generator=generator)
        for indices in order.split(128):
            optimizer.zero_grad()
            loss = nn.functional.mse_loss(model(X[indices]).squeeze(-1), y[indices])
            assert torch.isfinite(loss).item(), 'Non-finite training loss'
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(indices)
        return total_loss / len(X)

    def fit(self, X, y):
        X, y = validate_data(
            self, X, y, dtype=np.float32, ensure_all_finite='allow-nan',
            y_numeric=True,
        )
        assert np.isfinite(y).all(), 'Targets must be finite'
        inner_train, inner_valid = train_test_split(
            np.arange(len(X)), test_size=0.15, random_state=self.random_state,
        )
        preprocessing = self._preprocessor()
        inner_X = self._tensor(preprocessing.fit_transform(X[inner_train]))
        valid_X = self._tensor(preprocessing.transform(X[inner_valid]))
        inner_y = self._tensor(y[inner_train])
        valid_y = self._tensor(y[inner_valid])
        self.history_ = {
            'inner_training_mse': [], 'inner_validation_mse': [],
            'refit_training_mse': [],
        }

        with torch.random.fork_rng(devices=[]):
            model, optimizer, generator = self._new_network()
            best_loss = np.inf
            epochs_without_improvement = 0
            for epoch in range(1, 301):
                train_loss = self._train_epoch(
                    model, optimizer, generator, inner_X, inner_y,
                )
                model.eval()
                with torch.no_grad():
                    validation_loss = nn.functional.mse_loss(
                        model(valid_X).squeeze(-1), valid_y,
                    ).item()
                assert np.isfinite(validation_loss), 'Non-finite validation loss'
                self.history_['inner_training_mse'].append(train_loss)
                self.history_['inner_validation_mse'].append(validation_loss)
                if validation_loss < best_loss - 0.000001:
                    best_loss = validation_loss
                    self.best_epoch_ = epoch
                    epochs_without_improvement = 0
                else:
                    epochs_without_improvement += 1
                if epochs_without_improvement >= 25:
                    break

            self.preprocessing_ = self._preprocessor()
            full_X = self._tensor(self.preprocessing_.fit_transform(X))
            full_y = self._tensor(y)
            model, optimizer, generator = self._new_network()
            for _ in range(self.best_epoch_):
                self.history_['refit_training_mse'].append(self._train_epoch(
                    model, optimizer, generator, full_X, full_y,
                ))
            model.eval()
            self.model_ = model
        return self

    def predict(self, X):
        check_is_fitted(self, ['model_', 'preprocessing_'])
        X = validate_data(
            self, X, reset=False, dtype=np.float32,
            ensure_all_finite='allow-nan',
        )
        transformed = self._tensor(self.preprocessing_.transform(X))
        self.model_.eval()
        with torch.no_grad():
            predictions = self.model_(transformed).squeeze(-1).numpy()
        assert np.isfinite(predictions).all(), 'Non-finite predictions'
        return predictions

In [ ]:
synthetic_rng = np.random.default_rng(42)
synthetic_X = synthetic_rng.normal(size=(48, 3))
synthetic_y = 0.5 + 0.2 * synthetic_X[:, 0] - 0.1 * synthetic_X[:, 1]
synthetic_X[::5, 1] = np.nan

try:
    TorchRegressor().predict(synthetic_X)
except NotFittedError:
    pass
else:
    raise AssertionError('predict must reject an unfitted estimator')

synthetic_model = TorchRegressor().fit(synthetic_X, synthetic_y)
synthetic_predictions = synthetic_model.predict(synthetic_X)
assert synthetic_predictions.shape == (len(synthetic_X),)
assert np.isfinite(synthetic_predictions).all()
assert mean_squared_error(synthetic_y, synthetic_predictions) < np.var(synthetic_y)
repeated_predictions = TorchRegressor().fit(synthetic_X, synthetic_y).predict(synthetic_X)
np.testing.assert_array_equal(synthetic_predictions, repeated_predictions)
assert 1 <= synthetic_model.best_epoch_ <= 300
assert len(synthetic_model.history_['refit_training_mse']) == synthetic_model.best_epoch_
print('Neural-network self-check passed.')
synthetic_statistics = pd.DataFrame({'severity_sum': synthetic_y, 'collision_count': np.ones(len(synthetic_y))})
wrapped_network = FoldTargetRegressor(TorchRegressor()).fit(synthetic_X, synthetic_statistics)
assert wrapped_network.predict(synthetic_X).shape == synthetic_y.shape
assert np.isfinite(wrapped_network.score(synthetic_X, synthetic_statistics))
assert np.isfinite(wrapped_network.regressor_.history_['refit_training_mse']).all()
print('Fold-local neural-network self-check passed.')


In [ ]:
neural_network_search = GridSearchCV(
    FoldTargetRegressor(TorchRegressor(), target_name=selected_target),
    param_grid={
        'regressor__hidden_sizes': [(32,), (64,), (64, 32)],
        'regressor__weight_decay': [0, 0.001],
    },
    cv=model_cv,
    scoring=None,
    n_jobs=1,
    error_score='raise',
    return_train_score=True,
)
neural_network_search.fit(X_train, y_train)

neural_network_cv = pd.DataFrame(neural_network_search.cv_results_)
assert len(neural_network_cv) == 6
fold_columns = [f'split{fold}_test_score' for fold in range(5)]
assert len(neural_network_cv.filter(regex=r'^split\d+_test_score$').columns) == 5
assert np.isfinite(neural_network_cv[fold_columns].to_numpy()).all()
display(neural_network_cv[[
    'param_regressor__hidden_sizes', 'param_regressor__weight_decay', 'mean_test_score', 'std_test_score',
]].sort_values('mean_test_score', ascending=False).rename(columns={
    'mean_test_score': 'mean_cv_r2', 'std_test_score': 'std_cv_r2',
}))

neural_network_model = evaluate_regressor('PyTorch MLP', neural_network_search)
assert sum(result['model'] == 'PyTorch MLP' for result in model_results) == 1
print('Epochs selected before full-training refit:', neural_network_model.regressor_.best_epoch_)

### 14. Compare models without selecting on the test set

In [ ]:
model_comparison = (
    pd.DataFrame(model_results)
    .sort_values('cv_r2', ascending=False)
    .reset_index(drop=True)
)
selected_model_name = model_comparison.loc[0, 'model']
selected_model = fitted_models[selected_model_name]

print(f'Selected model from cross-validation: {selected_model_name}')
model_comparison

In [ ]:
comparison_by_name = model_comparison.set_index('model')
neural_network_deltas = (
    comparison_by_name.loc['PyTorch MLP', ['cv_r2', 'test_r2']]
    - comparison_by_name.loc[['LightGBM', 'XGBoost'], ['cv_r2', 'test_r2']]
).rename(columns={'cv_r2': 'nn_minus_tree_cv_r2', 'test_r2': 'nn_minus_tree_test_r2'})
display(neural_network_deltas)

for tree_name, deltas in neural_network_deltas.iterrows():
    cv_outcome = 'surpasses' if deltas['nn_minus_tree_cv_r2'] > 0 else 'ties' if deltas['nn_minus_tree_cv_r2'] == 0 else 'loses to'
    test_outcome = 'surpasses' if deltas['nn_minus_tree_test_r2'] > 0 else 'ties' if deltas['nn_minus_tree_test_r2'] == 0 else 'loses to'
    print(
        f'PyTorch MLP {cv_outcome} {tree_name} on CV R² '
        f'({deltas["nn_minus_tree_cv_r2"]:+.4f}) and {test_outcome} it on test R² '
        f'({deltas["nn_minus_tree_test_r2"]:+.4f}).'
    )
print('Model selection uses CV R². These comparisons cover one historical split and seed.')

In [ ]:
model_comparison.set_index('model')[['cv_r2', 'test_r2']].plot(kind='bar')
plt.title(f'Regression Model Comparison: {selected_target}')
plt.ylabel('R²')
plt.xticks(rotation=25, ha='right')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## Takeaways

- Corrected full-search results are pending. Previous scores, chosen configurations, and model rankings were removed because the smoothing prior included validation-fold labels.
- Both targets and all six regressors remain available. Target selection and model ranking use training cross-validation; held-out metrics remain final diagnostics.
- Each outer cross-validation fit learns its own smoothing prior. The neural network selects epochs within that training partition, then restarts and refits on every row supplied to the fit.
- Direct severity components and the former risk target remain excluded from predictors and enforced with an assertion.
- Small self-checks establish calculation and search compatibility, not predictive performance. A full rerun is needed before reporting new comparisons.
- The random historical cell split does not establish future forecasting performance or superiority across seeds.
